# Berkeley Haas Exec Ed
# Artificial Intelligence: Business Strategies and Applications
## Module 3 - Neural Networks and Deep Learning
## Building and deploying an image classification app using a pretrained Google deep learning model and deploying it on Hugging Face Spaces
James Gray, Learning Facilitator (jamesgray@berkeley.edu)

**Instructor**: James Gray
**Web**: http://www.jamesgray.ai
**Email**: james@jamesgray.ai
**Discord**: https://discord.jamesgray.ai

## What You Will Build in This Notebook

**Image classification** is one of the most impactful applications of AI in business today. It enables computers to automatically identify and categorize objects in images — a task that previously required human reviewers.

### Real-World Business Applications
- **Manufacturing**: Automated quality control — detecting defects on assembly lines in real time
- **Healthcare**: Medical imaging analysis — identifying anomalies in X-rays, MRIs, and pathology slides
- **Retail**: Visual search and inventory management — recognizing products from photos
- **Content Moderation**: Automatically flagging inappropriate images across platforms

### What We'll Do Today
In this notebook, you will:
1. Load a **pre-trained deep learning model** (Google's Vision Transformer) that has already learned to recognize 1,000 categories of objects
2. Walk through how the model processes and classifies an image, step by step
3. Build a **web application** where you can upload any image and get an instant AI classification

**No coding experience is required** — the goal is to understand *how* these models work and *what* they can do for your business, not to memorize the code.

## 1 - Install the Python packages for this demo

Before we start, we need to download a few **packages** — pre-built toolkits that give Python new capabilities (similar to installing an app on your phone).

- **transformers**: Created by Hugging Face, this package lets us download and use pre-trained AI models with just a few lines of code
- **gradio**: Lets us build a simple web interface so anyone can upload images and see results — no coding required
- **torch & torchvision**: PyTorch, the deep learning framework that powers the model's computations behind the scenes

The `!pip install` command below downloads and installs all of these at once. The `-q` flag just means "quiet" — it reduces the installation output to keep things clean.

In [ ]:
!pip install -q transformers gradio torch torchvision
# transformers: Hugging Face library for pre-trained AI models
# gradio: build simple web UIs for ML models
# torch & torchvision: PyTorch deep learning framework and image utilities

## 2 - Import Python libraries from the packages installed above

Now that the packages are installed, we need to **import** the specific tools we'll use — think of it like opening a toolbox and pulling out the tools you need for the job.

We're importing four things:
- **ViTImageProcessor**: Prepares images for the model (resizing, formatting, etc.)
- **ViTForImageClassification**: The actual AI model — Google's Vision Transformer, pre-trained on 14 million images across 21,843 categories at 224×224 resolution. Learn more at the [Hugging Face model card](https://huggingface.co/google/vit-base-patch16-224)
- **PIL (Image)**: The Python Imaging Library — lets us open and work with image files
- **requests**: Lets us download images (or any file) from the internet via a URL

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image  # Python Imaging Library for manipulating images
import requests

## 3 - Download a sample image from the web

We need a test image to run through our classifier. In a real business scenario, this would be your own images — product photos, medical scans, security footage, etc.

For this demo, we'll use an image from the **COCO dataset**, a widely-used collection of everyday images created for AI research. Our sample image shows two cats sitting on a couch.

In [ ]:
url = 'https://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

In [ ]:
# Let's see the image we'll be classifying
print(f"Image size: {image.size[0]} x {image.size[1]} pixels")
image

## What is a Pre-trained Model?

Think of a pre-trained model like **hiring an expert who already has years of experience**. Instead of training someone from scratch (which takes enormous time and data), you bring in someone who already knows the fundamentals and can immediately start contributing.

Google's Vision Transformer (ViT) was trained on **14 million images** across thousands of categories. That training took massive computational resources — but we can download and use the finished model in seconds.

### Why This Matters for Business
- **Cost**: Training a model from scratch can cost millions of dollars in compute. Using a pre-trained model is essentially free.
- **Speed**: Go from idea to working prototype in hours, not months.
- **Accessibility**: Your team doesn't need a PhD in machine learning — pre-trained models are available through simple APIs.
- **Customization**: Through a technique called *transfer learning*, you can adapt a pre-trained model to your specific use case with relatively little data (e.g., training it to recognize *your* products or *your* defect types).

## 4 - Configure the Image Processor

Before our image can be classified, it needs to be prepared in exactly the format the model expects — think of it like filling out a form where every field must be in the right format, or it gets rejected.

In this step, we download the **preparation instructions** (not the model itself — that comes in Step 5). These instructions tell the processor how to:
- **Resize** the image to 224×224 pixels (the exact size the model was trained on)
- **Normalize** pixel values (rescale brightness to a standard range)
- **Convert** the image into the numerical format the model requires

`from_pretrained('google/vit-base-patch16-224')` downloads these settings directly from Hugging Face, ensuring they match the specific model we'll use.

We'll put this processor to work in **Step 6**, where it actually transforms our image.

In [ ]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

## 5 - Download the Pre-trained Classification Model

In Step 4, we downloaded the *preparation instructions*. Now we download the **model itself** — the AI "brain" that actually makes predictions.

This is Google's Vision Transformer (ViT), fine-tuned on **ImageNet** — a dataset of 1,000 everyday object categories (like "cat", "dog", "airplane", "coffee mug"). The model has learned patterns from millions of images and can now recognize these objects in photos it has never seen before.

`from_pretrained` downloads the model's learned knowledge (millions of numerical parameters that encode what the model has learned about visual patterns) from Hugging Face's Model Hub. This model can be:
- Used **immediately** for predictions (which is what we'll do)
- **Fine-tuned** on your own data for specialized tasks (e.g., classifying your specific products)

In [ ]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

## 6 - Process the Image into a Tensor

When you look at a photo, you instantly see shapes, colors, and objects. A computer doesn't — it can only work with numbers. So before the model can classify our image, we need to **translate the photo into a structured grid of numbers**. This grid is called a **tensor** (think of it like a spreadsheet, but with multiple layers).

Here's what the processor we configured in Step 4 does behind the scenes:
1. **Resizes** the image to exactly 224×224 pixels (the size the model was trained on)
2. **Breaks the image into its Red, Green, and Blue color channels** — three separate layers of pixel intensity values
3. **Normalizes** the pixel values — instead of the usual 0–255 brightness scale, it rescales them to a smaller range that helps the model learn more effectively

The code parameters:
- `images=image` — the photo we downloaded in Step 3
- `return_tensors="pt"` — return the result as a PyTorch tensor ("pt" = PyTorch)

The result (`inputs`) is a Python dictionary containing the fully processed image data, ready to be fed into the model.

In [ ]:
inputs = processor(images=image, return_tensors="pt")

## 7 - Inspect the Tensor to Understand What the Model "Sees"

In the previous step, we converted our photo into numbers. Now let's look at the result to understand what the model actually receives as input. This is useful because it demystifies what's happening "under the hood" — the model never sees the photo itself, only this grid of numbers.

- `inputs` is a Python dictionary containing the processed image data
- `pixel_values` contains the actual numbers — each one represents how bright a pixel is in a given color channel, scaled to a standard range (this scaling is called **normalization**, and it helps the model learn more effectively)

The tensor (think of it as a **spreadsheet with extra dimensions**) has 4 dimensions:
- **Dimension 1** (size 1): batch size — we're classifying one image at a time
- **Dimension 2** (size 3): the three color channels — every digital image is made of Red, Green, and Blue (RGB) layers blended together
- **Dimension 3** (size 224): image height in pixels (resized from the original)
- **Dimension 4** (size 224): image width in pixels (resized from the original)

So our single image becomes **3 × 224 × 224 = 150,528 numbers** — that's what the model processes to make its prediction.

In [ ]:
print("Tensors:", inputs)
# lets see what the shape is of the pixel_values tensor
tensor = inputs['pixel_values']
print ("The tensor shape for the dimensions are;", tensor.shape)

## 8 - Run the Model to Predict the Image Class

This is the moment of truth — we feed our processed image into the model and ask it to make a prediction. Behind the scenes, the model analyzes the 150,528 numbers from Step 7, looking for patterns it recognizes from its training.

The model returns **logits** — think of these as raw "scores" the model assigns to each of the 1,000 possible categories. A higher score means the model thinks the image is more likely to be that category. These aren't percentages yet — we'll convert them to confidence scores in Step 12.

In [ ]:
outputs = model(**inputs)
print("Model outputs:", outputs)

## 9 - Extract the Logits (Raw Prediction Scores)

In Step 8, the model returned a large data structure with metadata. Now we pull out just the **logits** — the 1,000 scores that represent the model's prediction.

When you run the code below, you'll see a long list of numbers — one for each of the 1,000 categories the model knows. Most values will be small or negative (meaning "probably not this category"), and a few will be large and positive (meaning "this could be a match").

The shape `(1, 1000)` tells us: **1 image**, **1,000 scores** — one score per category.

In [ ]:
logits = outputs.logits
print("Logits:", logits)
# the logits data structure is a 1D array of 1 row with 1000 columns that store the prediction for the class
print (logits.shape)

## 10 - Find the Predicted Class

Out of the 1,000 scores, we need to find the **winner** — the category with the highest score. That's what `argmax` does: it scans all 1,000 values and returns the position of the largest one.

- `logits.argmax(-1)` — finds the position of the highest score (e.g., position 285)
- `.item()` — converts the result to a simple number

The result is an index number (e.g., 285) — but what *is* category 285? We'll look that up in the next step.

In [ ]:
predicted_class_idx = logits.argmax(-1).item()
# this returns an "index" into the 1D array of the maximum value. Let's see what number it returns in the array
print("The index of the predicted class is column:", predicted_class_idx)

## 11 - Translate the Predicted Class into a Readable Name

The model predicted category **285** — but what does that mean? The model comes with a built-in **lookup table** (`model.config.id2label`) that maps every index number to a human-readable name.

This is like looking up a product code in a catalog — index 285 maps to a specific label that tells us what the model "sees" in the image.

You can browse the full list of all 1,000 categories in the model's configuration file on Hugging Face:
[View config.json on Hugging Face](https://huggingface.co/google/vit-base-patch16-224/blob/main/config.json)

Let's first peek at a sample of the lookup table to see what it looks like, then translate our prediction.

In [ ]:
# Let's peek at the lookup table — here are the first 10 categories and a few others
print("Sample entries from the lookup table (model.config.id2label):\n")
for idx in list(range(10)) + [285, 463, 817]:
    print(f"  Index {idx:4d} → {model.config.id2label[idx]}")
print(f"\nTotal categories: {len(model.config.id2label)}")

In [ ]:
# Translate the predicted class index into a human-readable class name
print("Predicted class:", model.config.id2label[predicted_class_idx])

## 12 - Understanding Confidence: The Model's Top 5 Predictions

The model doesn't just pick one answer — it assigns a **probability** to every possible class. This is critical for business applications: knowing that the model is 90% confident vs. 51% confident changes how you should act on its prediction.

We use a function called **softmax** to convert the raw logits into probabilities that sum to 100%.

**Business example**: If you're using AI to inspect products on a manufacturing line, you might automatically reject items where the model is 95%+ confident it found a defect, send items with 60–95% confidence to a human reviewer, and pass items below 60%. Setting these thresholds is a business decision, not a technical one.

In [ ]:
import torch

# Convert raw logits to probabilities using softmax
probabilities = torch.nn.functional.softmax(logits, dim=-1)

# Get the top 5 predictions with their confidence scores
top5_prob, top5_idx = torch.topk(probabilities, 5)

print("Top 5 Predictions:\n")
for i in range(5):
    label = model.config.id2label[top5_idx[0][i].item()]
    confidence = top5_prob[0][i].item() * 100
    print(f"  {i+1}. {label:30s} {confidence:.2f}%")

## Complete Code: End-to-End Image Classification

Here's everything from Steps 1–12 consolidated into a single code block. You can copy this cell into any Python environment to run the full classification pipeline.

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image  # Python Imaging Library for manipulating images
import requests
import torch

# Download a sample image from the web
url = 'https://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

# Configure the image processor and download the pre-trained model
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

# Process the image into a tensor
inputs = processor(images=image, return_tensors="pt")

# The tensor shape: [batch_size, channels (RGB), height (224), width (224)]
tensor = inputs['pixel_values']
print("Tensor shape:", tensor.shape)

# Run the model to get predictions
outputs = model(**inputs)
logits = outputs.logits

# Find the predicted class
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])

# Show confidence scores for top 5 predictions
probabilities = torch.nn.functional.softmax(logits, dim=-1)
top5_prob, top5_idx = torch.topk(probabilities, 5)
print("\nTop 5 Predictions:")
for i in range(5):
    label = model.config.id2label[top5_idx[0][i].item()]
    confidence = top5_prob[0][i].item() * 100
    print(f"  {i+1}. {label:30s} {confidence:.2f}%")

## 13 - Building a Web Application with Gradio

So far, we've been running code in a notebook. But in the real world, your end users — whether they're quality inspectors, radiologists, or content moderators — won't be writing Python code. They need a **simple, intuitive interface**.

**Gradio** is a Python library that lets you wrap any machine learning model in a web application with just a few lines of code. Users can upload images through their browser and instantly see results — no technical knowledge required.

This is a key step in making AI **operationally useful**: the gap between a working model and a deployed product is often the hardest part, and tools like Gradio dramatically simplify it.

In [ ]:
import gradio as gr

### Define the Classification Function

We wrap our classification logic into a single function that Gradio can call. This function takes an image, runs it through our model, and returns the top 5 predictions with confidence scores.

In [ ]:
import torch

# Build a function that classifies an image and returns the top 5 predictions with confidence scores
def get_image_classification(pil_image):
    inputs = processor(images=pil_image, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    # Return a dictionary of label: confidence for Gradio
    top5_prob, top5_idx = torch.topk(probabilities, 5)
    return {model.config.id2label[top5_idx[0][i].item()]: top5_prob[0][i].item() for i in range(5)}

### Build the Web Interface

`gr.Interface` connects our function to a web UI. We specify:
- **inputs**: an image upload widget
- **outputs**: a label display showing the top 5 classes with confidence bars
- **title/description**: what users see when they open the app

In [ ]:
# Build the Gradio web interface
demo = gr.Interface(
    fn=get_image_classification,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="AI Image Classifier",
    description="Upload any image to classify it using Google's Vision Transformer (ViT). The model will return its top 5 predictions with confidence scores."
)

In [ ]:
# Let's launch the web app!
demo.launch()

## Conclusion & Key Takeaways

### What We Accomplished
In this notebook, you built a complete **AI-powered image classification application** — from loading a pre-trained model to deploying a web interface that anyone can use.

### Key Takeaways for Business Leaders

1. **Pre-trained models are a game-changer**: You don't need to build AI from scratch. Models trained on millions of images are freely available and can be deployed in minutes.

2. **AI predictions are probabilistic**: Models don't give yes/no answers — they provide confidence scores. Understanding this is critical for designing business processes around AI (e.g., setting confidence thresholds for automated decisions vs. human review).

3. **The gap from model to product is shrinking**: Tools like Gradio and Hugging Face make it possible to go from a working model to a usable application in hours, not months.

4. **Domain expertise still matters**: The model can classify images, but deciding *which* images to classify, *what* confidence threshold to use, and *how* to act on the results — that's where business judgment is essential.

### Next Steps
- **Fine-tuning**: Adapt the model to your specific use case (e.g., classifying your products, your defects, your documents) using transfer learning
- **Deployment**: Publish your app to [Hugging Face Spaces](https://huggingface.co/spaces) for free cloud hosting
- **Evaluation**: Learn how to measure model performance (accuracy, precision, recall) to make informed deployment decisions